# Metacognitive Medical Digital Twins Pipeline

This notebook implements the complete MDT pipeline with MIMIC-IV and Medical-O1 data sources.

**Alignment Components:**
- Theory of Mind module for user belief inference
- Composite reward engine (5 components: safety, empathy, proactivity, metacognition, semantic)
- GRPO training for multi-objective alignment

**Ontology Components:**
- LOINC, SNOMED-CT, ICD-10 code mappings
- Clinical reference ranges validation
- MIMIC-IV item ID mappings

**Data Sources:**
- MIMIC-IV (ICU trajectories)
- Medical-O1 (reasoning chains)


## 1. Setup & Environment

In [ ]:
# Environment setup
import sys
sys.path.append(".")

# Core imports
from config.configs import DataConfig
from data.mimic_processor import MIMICProcessor
from data.medical_o1_processor import MedicalO1Processor
from core.theory_of_mind import TheoryOfMindModule
from rewards.composite_engine import CompositeRewardEngine
from training.grpo_trainer import run_grpo_training
from utils.ontology_validator import OntologyValidator
from utils.helpers import clean_memory, setup_logging

print("✓ All components imported successfully")


## 2. Configuration

In [ ]:
# Setup logging
setup_logging()
import logging
logger = logging.getLogger(__name__)

logger.info("="*80)
logger.info("MEDICAL DIGITAL TWIN - MASTER PIPELINE")
logger.info("="*80)

# Load configuration
data_config = DataConfig()
print(f"✓ Configuration loaded")
print(f"  MIMIC patients: {data_config.max_patients}")
print(f"  Medical-O1 examples: {data_config.max_o1_examples}")


## 3. Data Loading

In [ ]:
# Load MIMIC-IV data
print("Loading MIMIC-IV data...")
mimic_processor = MIMICProcessor(data_config)

if mimic_processor.check_availability():
    mimic_data = mimic_processor.process_all_patients(
        max_patients=data_config.max_patients
    )
    print(f"✓ Loaded {len(mimic_data)} MIMIC examples")
    
    # Load ICD diagnoses for validation
    try:
        icd_diagnoses = mimic_processor.load_icd_diagnoses()
        print(f"✓ Loaded {len(icd_diagnoses)} ICD diagnoses")
    except Exception as e:
        print(f"⚠️ ICD diagnoses loading failed: {e}")
        icd_diagnoses = None
else:
    print("⚠️ MIMIC data not available")
    mimic_data = []
    icd_diagnoses = None

## 4. Model Training (SFT)

## 3.5. Ontology Validation


In [ ]:
# Initialize ontology validator
print("Initializing ontology validator...")
ontology_validator = OntologyValidator()

# Validate ICD codes if available
if icd_diagnoses is not None:
    print("Validating ICD-10 codes...")
    valid_icd_count = 0
    total_icd_count = len(icd_diagnoses)
    
    # Sample validation (check first 100 for speed)
    sample_icd = icd_diagnoses.head(100)
    for _, row in sample_icd.iterrows():
        icd_code = row['icd_code']
        if ontology_validator.validate_icd10_code(icd_code):
            valid_icd_count += 1
    
    print(f"✓ ICD-10 validation: {valid_icd_count}/{len(sample_icd)} sample codes valid")
    print(f"  Total ICD diagnoses available: {total_icd_count}")
else:
    print("⚠️ ICD diagnoses not available for validation")

print("✓ Ontology validation completed")

In [ ]:
# Import training components
from training.sft_trainer import run_sft_training
from models.mdt_model import MedicalDigitalTwinModel
from config.configs import ModelConfig

# Initialize model configuration
model_config = ModelConfig()
print(f"✓ Model config loaded: {model_config.model_name}")

# Initialize model
model = MedicalDigitalTwinModel(model_config)

# Prepare training data split
if len(all_training_data) > 0:
    # Split data for training and evaluation (90% train, 10% eval)
    split_idx = int(len(all_training_data) * 0.9)
    train_data = all_training_data[:split_idx]
    eval_data = all_training_data[split_idx:]
    print(f"✓ Data split: {len(train_data)} train, {len(eval_data)} eval examples")
else:
    train_data = []
    eval_data = []
    print("⚠️ No training data available")

# Run SFT training
print("Starting SFT training...")
if len(train_data) > 0:
    trained_model = run_sft_training(
        model=model,
        train_dataset=train_data,
        eval_dataset=eval_data,
        config=data_config
    )
    print("✓ SFT training completed")
else:
    print("⚠️ Skipping SFT training - no data available")
    trained_model = model

## 5. Alignment Training (GRPO)

In [ ]:
# Import GRPO training components
from training.grpo_trainer import run_grpo_training
from config.configs import GRPOConfig
from data.dataset import CognitiveStreamDataset
from torch.utils.data import DataLoader

# Initialize reward engine
reward_engine = CompositeRewardEngine()

# Create GRPO configuration
grpo_config = GRPOConfig()
print(f"✓ GRPO config loaded: {grpo_config.num_iterations} iterations")

# Create dataloader for GRPO training
if len(train_data) > 0:
    # Create dataset
    grpo_dataset = CognitiveStreamDataset(
        data=train_data,
        tokenizer=trained_model.tokenizer,
        max_length=model_config.max_length
    )

    # Create dataloader
    grpo_dataloader = DataLoader(
        grpo_dataset,
        batch_size=grpo_config.batch_size,
        shuffle=True
    )
    print(f"✓ Created GRPO dataloader with {len(grpo_dataset)} examples")
else:
    grpo_dataloader = None
    print("⚠️ No training data available for GRPO")

# Run GRPO alignment
print("Starting GRPO alignment training...")
if grpo_dataloader is not None:
    aligned_model = run_grpo_training(
        model=trained_model,
        train_dataloader=grpo_dataloader,
        config=grpo_config,
        reward_engine=reward_engine
    )
    print("✓ GRPO alignment completed")
else:
    print("⚠️ Skipping GRPO training - no dataloader available")
    aligned_model = trained_model

## 6. Evaluation

In [ ]:
# Import evaluation
from evaluation.evaluator import MedicalTwinEvaluator

# Run evaluation
evaluator = MedicalTwinEvaluator()
results = evaluator.evaluate_model(aligned_model)

print("✓ Evaluation completed")
print(f"Results: {results}")
